# Notebook 03: Medical Vision Encoder and Abnormality Classification
## Hallucination-Resistant Medical Report Generation Using Generative AI

### Research Objectives:
1. Load pretrained medical vision backbone (Vision Transformer ViT-B/16 or TorchXRayVision DenseNet).
2. Extract dense spatial visual embeddings and global image representation vectors.
3. Compute calibrated multi-label abnormality classification probabilities across 10 target categories.
4. Perform batch inference on chest X-ray validation set with GPU acceleration.


In [ ]:
# 1. Module Imports and Hardware Setup
import sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/hallucination_medical_ai')]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
from src.config import get_config
from src.vision.vision_encoder import MedicalVisionEncoder
from src.vision.abnormality_classifier import AbnormalityClassifier

cfg = get_config()
print(f'Running Vision Transformer pipeline on: {cfg.env.device}')


### 2. Instantiate Medical Vision Encoder and Extract Feature Embeddings


In [ ]:
encoder = MedicalVisionEncoder(cfg)
print('Encoder backbone:', encoder.backbone_name)

sample_tensor = torch.randn(2, 3, 224, 224).to(cfg.env.device)
with torch.no_grad():
    features = encoder.forward_features(sample_tensor)
print('Feature embedding shape:', features.shape)


### 3. Abnormality Classification and Probability Prediction
Predict probabilities across target conditions (Cardiomegaly, Effusion, Pneumothorax, Consolidation, Atelectasis, Edema, etc.).


In [ ]:
classifier = AbnormalityClassifier(cfg, encoder=encoder)
preds = classifier.predict(sample_tensor)
print('Predicted probabilities for sample 0:')
for abnormality, prob in preds[0].items():
    print(f'  {abnormality:25s}: {prob:.4f}')


### 4. Visual Evidence Representation for Downstream Verification
Capture both top-level findings and fine-grained feature representations.


In [ ]:
evidence = classifier.get_visual_evidence(sample_tensor[0:1])
print('Evidence Summary:')
print('Detected abnormalities:', evidence['detected_abnormalities'])
print('Uncertain range findings:', evidence['uncertain_abnormalities'])
